# 1-Week Recursive Forecast Engine (168-Hour Prediction)

This notebook generates **station-level recursive forecasts for the next 168 hours (1 week)** using the production-ready XGBoost model trained for direct net flow prediction.

The engine combines:

- the latest reliable hourly station state
- recent historical net flow dynamics
- future hourly weather
- future event activity
- cyclic calendar features

to recursively estimate, for each future hour:

- net flow
- projected bikes available
- projected docks available
- station-level operational risk

The main output is:

- `workspace.default.current_station_predictions_multihour_netflow_direct_v3_cyclic_stationtrend`

This notebook represents the **multi-hour forecasting engine** of the project.

## Process Overview

This notebook performs the following steps:

### 1. Load model artifacts and metadata
The notebook reads:
- feature metadata
- best-model metadata
- the selected XGBoost model

This guarantees consistency between training and forecasting.

### 2. Read the latest operational station flow table
The current station-hour operational dataset is loaded from:
- `station_hour_flow_current`

Only reliable historical records are retained to define the latest valid system state.

### 3. Identify the latest closed hour
The engine uses the most recent high-quality hour as the starting point for the recursive multi-hour simulation.

### 4. Rebuild recent historical features
Historical net flow features are reconstructed, including:
- lag features
- rolling net flow statistics
- station-level recent activity metrics

These provide the model with the required initial state.

### 5. Load future weather
Future hourly weather is read from:
- `vw_weather_hourly_minimal_range_future`

This provides forward-looking environmental context for each forecast hour.

### 6. Precompute future event features
Future public events are converted into station-level hourly influence features using:
- event timing
- event attendance
- geographic distance
- spillover intensity logic

This avoids recomputing event logic at every recursive step.

### 7. Run the recursive forecast loop
For each hour in the 168-hour horizon, the notebook:
- builds the calendar features
- injects future weather
- injects future event features
- updates lag and rolling net flow variables recursively
- scores the model
- updates the projected station state

### 8. Convert predictions into operational station forecasts
For each future hour, the notebook estimates:
- projected bikes available
- projected docks available
- station risk label

### 9. Save the final multi-hour forecast table
The full 168-hour station-level forecast output is written into a Delta table for downstream operational use and validation.

In [0]:
%pip install xgboost==2.0.3
%restart_python

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import pandas as pd
import numpy as np
import json
import zlib
import xgboost as xgb

# ============================================================
# MULTI-HOUR FORECAST ENGINE
# Model: V3_DIRECT_NETFLOW_CYCLIC_STATIONTREND
# Recursive forecast for next 168 hours
# Uses:
#   - historical station flow from station_hour_flow_current
#   - future weather from vw_weather_hourly_minimal_range_future
#   - future events from public_events_hourly_detail
#
# MODIFIED VERSION:
#   Adds event metadata to output for Power BI:
#     - primary_event_id
#     - primary_event_name
#     - primary_event_category
#     - primary_event_distance_km
#     - impacting_event_ids
#     - impacting_event_names
#
# IMPORTANT:
#   This does NOT change model features nor prediction logic.
#   It only enriches the output table with event-identification columns.
# ============================================================

# ------------------------------------------------------------
# 0) CONFIG
# ------------------------------------------------------------
FLOW_TBL = "workspace.default.station_hour_flow_current"

WEATHER_FUTURE_VIEW = "workspace.default.vw_weather_hourly_minimal_range_future"

EVENTS_HOURLY_DETAIL_DIR = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver_agg/"
    "public_events_hourly_detail"
)

FEATURE_META_JSON_DBFS = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata/"
    "netflow_direct_v3_cyclic_stationtrend/features_netflow_direct_v3_cyclic_stationtrend.json"
)

BEST_MODEL_META_JSON_DBFS = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata/"
    "netflow_direct_v3_cyclic_stationtrend/best_model_netflow_direct_v3_cyclic_stationtrend.json"
)

MULTI_FORECAST_OUT_TBL = (
    "workspace.default.current_station_predictions_multihour_netflow_direct_v3_cyclic_stationtrend"
)

HORIZON_HOURS = 168  # 1 week
LOOKBACK_HOURS = 220
MAX_MODEL_BYTES = 30_000_000
MAX_META_BYTES = 500_000

DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX = 43.63, 43.67
DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX = -79.41, -79.37

# -------- Event influence config --------
EVENT_RADIUS_KM = 3.0
EVENT_DECAY_KM = 1.5
EVENT_EPS = 0.10

VERY_HIGH_RADIUS_KM = 0.3
HIGH_RADIUS_KM = 0.6
MEDIUM_RADIUS_KM = 1.5
LOW_RADIUS_KM = 3.0

VERY_HIGH_MULT = 1.8
HIGH_MULT = 1.4
MEDIUM_MULT = 1.0
LOW_MULT = 0.6

PI = 3.141592653589793

# ------------------------------------------------------------
# 1) HELPERS
# ------------------------------------------------------------
def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def read_dbfs_text(dbfs_path: str, max_bytes: int) -> str:
    return dbutils.fs.head(dbfs_path, max_bytes)

def load_xgb_booster_from_dbfs_json(dbfs_path: str, max_bytes: int = MAX_MODEL_BYTES) -> xgb.Booster:
    s = read_dbfs_text(dbfs_path, max_bytes=max_bytes)
    if len(s) < 1000:
        raise Exception(f"Model JSON looks too small or truncated: {dbfs_path}")
    booster = xgb.Booster()
    booster.load_model(bytearray(s.encode("utf-8")))
    return booster

def station_to_bucket(station_id: str, n_buckets: int) -> int:
    if station_id is None:
        return 0
    return zlib.crc32(str(station_id).encode("utf-8")) % n_buckets

def cyc_hour(hour):
    angle = 2.0 * np.pi * hour / 24.0
    return np.sin(angle), np.cos(angle)

def cyc_dow(dow_num):
    dow_idx = dow_num - 1
    angle = 2.0 * np.pi * dow_idx / 7.0
    return np.sin(angle), np.cos(angle)

def cyc_month(month):
    month_idx = month - 1
    angle = 2.0 * np.pi * month_idx / 12.0
    return np.sin(angle), np.cos(angle)

def risk_label(pred_bikes, pred_docks):
    if pred_bikes <= 1:
        return "CRITICAL_EMPTY"
    elif pred_bikes <= 3:
        return "LOW_BIKES"
    elif pred_docks <= 1:
        return "CRITICAL_FULL"
    elif pred_docks <= 3:
        return "LOW_DOCKS"
    else:
        return "NORMAL"

def haversine_matrix_km(st_lat, st_lon, ev_lat, ev_lon):
    """
    Vectorized distance matrix:
      stations: N
      events  : M
    returns matrix (N, M) in km
    """
    st_lat = np.radians(np.asarray(st_lat, dtype=np.float64))[:, None]
    st_lon = np.radians(np.asarray(st_lon, dtype=np.float64))[:, None]
    ev_lat = np.radians(np.asarray(ev_lat, dtype=np.float64))[None, :]
    ev_lon = np.radians(np.asarray(ev_lon, dtype=np.float64))[None, :]

    dlat = ev_lat - st_lat
    dlon = ev_lon - st_lon

    a = np.sin(dlat / 2.0) ** 2 + np.cos(st_lat) * np.cos(ev_lat) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    return 6371.0 * c

def tier_multiplier_matrix(dist_km):
    """
    Piecewise multiplier by distance:
      <= 0.3 km : very high
      <= 0.6 km : high
      <= 1.5 km : medium
      <= 3.0 km : low
      >  3.0 km : 0
    """
    return np.select(
        [
            dist_km <= VERY_HIGH_RADIUS_KM,
            dist_km <= HIGH_RADIUS_KM,
            dist_km <= MEDIUM_RADIUS_KM,
            dist_km <= LOW_RADIUS_KM
        ],
        [
            VERY_HIGH_MULT,
            HIGH_MULT,
            MEDIUM_MULT,
            LOW_MULT
        ],
        default=0.0
    )

def build_precomputed_event_features(pdf_state_base: pd.DataFrame,
                                     pdf_events_detail: pd.DataFrame,
                                     latest_closed_ts,
                                     horizon_hours: int) -> dict:
    """
    Precompute event features once for all forecast hours and all stations.
    Returns:
      event_features_by_ts[target_ts] = DataFrame indexed like pdf_state_base

    IMPORTANT:
      This function preserves the original event feature logic and only adds
      descriptive metadata columns for downstream dashboards.
    """
    event_features_by_ts = {}

    station_template = pd.DataFrame(index=pdf_state_base.index)
    station_template["lat"] = pd.to_numeric(pdf_state_base["lat"], errors="coerce").fillna(0.0)
    station_template["lon"] = pd.to_numeric(pdf_state_base["lon"], errors="coerce").fillna(0.0)

    def build_empty_tmp(idx):
        tmp = pd.DataFrame(index=idx)
        tmp["event_day_flag"] = 0
        tmp["events_day_count"] = 0
        tmp["event_day_attendance_sum"] = 0.0
        tmp["event_active_nearby_flag"] = 0
        tmp["events_nearby_count"] = 0
        tmp["nearest_event_km"] = 999.0
        tmp["event_weighted_intensity"] = 0.0
        tmp["event_attendance_est_sum_nearby"] = 0.0
        tmp["event_impact_score"] = 0.0

        # debug / monitoring
        tmp["event_spillover_score"] = 0.0
        tmp["event_attendance_sum_03km"] = 0.0
        tmp["event_attendance_sum_06km"] = 0.0
        tmp["event_attendance_sum_15km"] = 0.0
        tmp["event_attendance_sum_30km"] = 0.0

        # new descriptive fields (NOT model features)
        tmp["primary_event_id"] = None
        tmp["primary_event_name"] = None
        tmp["primary_event_category"] = None
        tmp["primary_event_distance_km"] = None
        tmp["impacting_event_ids"] = None
        tmp["impacting_event_names"] = None

        return tmp

    if pdf_events_detail.empty:
        for horizon in range(1, horizon_hours + 1):
            target_ts = pd.Timestamp(latest_closed_ts) + pd.Timedelta(hours=horizon)
            event_features_by_ts[target_ts] = build_empty_tmp(pdf_state_base.index)
        return event_features_by_ts

    pdf_events_detail = pdf_events_detail.copy()

    pdf_events_detail["target_ts_hour"] = pd.to_datetime(pdf_events_detail["target_ts_hour"])
    pdf_events_detail["start_date"] = pd.to_datetime(pdf_events_detail["start_date"]).dt.normalize()
    pdf_events_detail["end_date"] = pd.to_datetime(pdf_events_detail["end_date"]).dt.normalize()
    pdf_events_detail["attendance_est"] = pd.to_numeric(pdf_events_detail["attendance_est"], errors="coerce").fillna(0.0)
    pdf_events_detail["event_lat"] = pd.to_numeric(pdf_events_detail["event_lat"], errors="coerce")
    pdf_events_detail["event_lon"] = pd.to_numeric(pdf_events_detail["event_lon"], errors="coerce")

    if "event_name" not in pdf_events_detail.columns:
        pdf_events_detail["event_name"] = None
    if "event_category" not in pdf_events_detail.columns:
        pdf_events_detail["event_category"] = None

    day_unique = (
        pdf_events_detail[["start_date", "end_date", "event_id", "attendance_est"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    active_by_ts = {}
    for ts_key, grp in pdf_events_detail.groupby("target_ts_hour"):
        active_by_ts[ts_key] = grp[
            ["event_id", "event_name", "event_category", "event_lat", "event_lon", "attendance_est"]
        ].copy().reset_index(drop=True)

    st_lat = station_template["lat"].values
    st_lon = station_template["lon"].values

    for horizon in range(1, horizon_hours + 1):
        target_ts = pd.Timestamp(latest_closed_ts) + pd.Timedelta(hours=horizon)
        target_date = pd.Timestamp(target_ts.date())

        tmp = build_empty_tmp(pdf_state_base.index)

        # -------- day-level --------
        day_mask = (
            (day_unique["start_date"] <= target_date) &
            (day_unique["end_date"] >= target_date)
        )
        pdf_events_day = day_unique.loc[day_mask, ["event_id", "attendance_est"]]

        if not pdf_events_day.empty:
            tmp["event_day_flag"] = 1
            tmp["events_day_count"] = int(pdf_events_day["event_id"].nunique())
            tmp["event_day_attendance_sum"] = float(pdf_events_day["attendance_est"].sum())

        # -------- hour-level --------
        pdf_events_active = active_by_ts.get(target_ts, None)
        if pdf_events_active is not None and not pdf_events_active.empty:
            ev_lat = pdf_events_active["event_lat"].astype(float).values
            ev_lon = pdf_events_active["event_lon"].astype(float).values
            ev_att = pdf_events_active["attendance_est"].astype(float).values

            ev_ids = pdf_events_active["event_id"].astype("string").fillna("")
            ev_names = pdf_events_active["event_name"].astype("string").fillna("")
            ev_categories = pdf_events_active["event_category"].astype("string").fillna("")

            dist_km = haversine_matrix_km(st_lat, st_lon, ev_lat, ev_lon)

            within_03 = dist_km <= VERY_HIGH_RADIUS_KM
            within_06 = dist_km <= HIGH_RADIUS_KM
            within_15 = dist_km <= MEDIUM_RADIUS_KM
            within_30 = dist_km <= LOW_RADIUS_KM

            nearest_km = dist_km.min(axis=1)
            nearby_count = within_30.sum(axis=1)
            nearby_flag = (nearby_count > 0).astype(int)

            attendance_03 = within_03.astype(np.float64) @ ev_att
            attendance_06 = within_06.astype(np.float64) @ ev_att
            attendance_15 = within_15.astype(np.float64) @ ev_att
            attendance_30 = within_30.astype(np.float64) @ ev_att

            # Existing feature: inverse-distance weighting, within 3 km
            inv_dist_weighted = np.where(within_30, ev_att[None, :] / (dist_km + EVENT_EPS), 0.0)
            event_weighted_intensity = inv_dist_weighted.sum(axis=1)

            # Existing spillover logic: exponential decay + distance tier multiplier
            tier_mult = tier_multiplier_matrix(dist_km)
            spillover_component = np.where(
                within_30,
                ev_att[None, :] * np.exp(-dist_km / EVENT_DECAY_KM) * tier_mult,
                0.0
            )
            spillover_score = spillover_component.sum(axis=1)

            # Keep model feature compatible
            event_impact_score = spillover_score

            tmp["event_active_nearby_flag"] = nearby_flag
            tmp["events_nearby_count"] = nearby_count.astype(int)
            tmp["nearest_event_km"] = nearest_km.astype(float)
            tmp["event_weighted_intensity"] = event_weighted_intensity.astype(float)
            tmp["event_attendance_est_sum_nearby"] = attendance_30.astype(float)
            tmp["event_impact_score"] = event_impact_score.astype(float)

            # debug / monitoring
            tmp["event_spillover_score"] = spillover_score.astype(float)
            tmp["event_attendance_sum_03km"] = attendance_03.astype(float)
            tmp["event_attendance_sum_06km"] = attendance_06.astype(float)
            tmp["event_attendance_sum_15km"] = attendance_15.astype(float)
            tmp["event_attendance_sum_30km"] = attendance_30.astype(float)

            # -------- NEW DESCRIPTIVE EVENT METADATA --------
            primary_event_ids = []
            primary_event_names = []
            primary_event_categories = []
            primary_event_distances = []
            impacting_event_ids_list = []
            impacting_event_names_list = []

            for i in range(dist_km.shape[0]):
                mask = within_30[i]

                if mask.any():
                    nearby_idx = np.where(mask)[0]

                    # Principal event = nearest event within 3 km
                    nearest_local_idx = nearby_idx[np.argmin(dist_km[i, nearby_idx])]

                    primary_event_ids.append(str(ev_ids.iloc[nearest_local_idx]))
                    primary_event_names.append(str(ev_names.iloc[nearest_local_idx]) if str(ev_names.iloc[nearest_local_idx]) != "" else None)
                    primary_event_categories.append(str(ev_categories.iloc[nearest_local_idx]) if str(ev_categories.iloc[nearest_local_idx]) != "" else None)
                    primary_event_distances.append(float(dist_km[i, nearest_local_idx]))

                    impacting_ids = [str(ev_ids.iloc[j]) for j in nearby_idx if str(ev_ids.iloc[j]) != ""]
                    impacting_names = [str(ev_names.iloc[j]) for j in nearby_idx if str(ev_names.iloc[j]) != ""]

                    impacting_event_ids_list.append(" | ".join(impacting_ids) if impacting_ids else None)
                    impacting_event_names_list.append(" | ".join(impacting_names) if impacting_names else None)
                else:
                    primary_event_ids.append(None)
                    primary_event_names.append(None)
                    primary_event_categories.append(None)
                    primary_event_distances.append(None)
                    impacting_event_ids_list.append(None)
                    impacting_event_names_list.append(None)

            tmp["primary_event_id"] = primary_event_ids
            tmp["primary_event_name"] = primary_event_names
            tmp["primary_event_category"] = primary_event_categories
            tmp["primary_event_distance_km"] = primary_event_distances
            tmp["impacting_event_ids"] = impacting_event_ids_list
            tmp["impacting_event_names"] = impacting_event_names_list

        event_features_by_ts[target_ts] = tmp

    return event_features_by_ts

# ------------------------------------------------------------
# 2) LOAD METADATA + MODEL
# ------------------------------------------------------------
if not path_exists(FEATURE_META_JSON_DBFS):
    raise Exception(f"Feature metadata not found: {FEATURE_META_JSON_DBFS}")

if not path_exists(BEST_MODEL_META_JSON_DBFS):
    raise Exception(f"Best model metadata not found: {BEST_MODEL_META_JSON_DBFS}")

feat_meta = json.loads(read_dbfs_text(FEATURE_META_JSON_DBFS, MAX_META_BYTES))
best_meta = json.loads(read_dbfs_text(BEST_MODEL_META_JSON_DBFS, MAX_META_BYTES))

FEATURES = feat_meta["netflow_features"]
HASH_BUCKETS = int(feat_meta.get("hash_buckets", 512))
BEST_MODEL = best_meta["best_model"]

if BEST_MODEL != "xgboost":
    raise Exception(f"This Job D currently supports only xgboost. Best model found: {BEST_MODEL}")

MODEL_PATH = best_meta["xgb_model_path"]
booster = load_xgb_booster_from_dbfs_json(MODEL_PATH)

print("Loaded model and metadata")
print("Best model   :", BEST_MODEL)
print("Feature count:", len(FEATURES))
print("Density features in metadata:", [f for f in FEATURES if "density" in f.lower()])

# ------------------------------------------------------------
# 3) READ EVENTS DETAIL INPUT
# ------------------------------------------------------------
if not path_exists(EVENTS_HOURLY_DETAIL_DIR):
    raise Exception(f"Events detail input not found: {EVENTS_HOURLY_DETAIL_DIR}")

df_events_detail = spark.read.parquet(EVENTS_HOURLY_DETAIL_DIR)

required_event_cols = {
    "target_ts_hour",
    "event_id",
    "event_name",
    "event_category",
    "start_date",
    "end_date",
    "event_lat",
    "event_lon",
    "attendance_est"
}
missing_event_cols = sorted(list(required_event_cols - set(df_events_detail.columns)))
if missing_event_cols:
    raise Exception(f"{EVENTS_HOURLY_DETAIL_DIR} missing required columns: {missing_event_cols}")

pdf_events_detail = df_events_detail.select(
    "target_ts_hour",
    "event_id",
    "event_name",
    "event_category",
    "start_date",
    "end_date",
    "event_lat",
    "event_lon",
    "attendance_est"
).toPandas()

print("Events detail rows loaded:", len(pdf_events_detail))
print("Distinct event_ids in detail:", pdf_events_detail["event_id"].nunique() if not pdf_events_detail.empty else 0)

# ------------------------------------------------------------
# 4) READ CURRENT HOURLY FLOW
# ------------------------------------------------------------
df = spark.table(FLOW_TBL)

required_cols = {
    "station_id", "year", "month", "day", "hour",
    "estimated_departures", "estimated_arrivals",
    "temperature_2m_celsius", "apparent_temperature_celsius",
    "date", "dow_num", "is_weekend",
    "event_day_flag", "events_day_count", "event_day_attendance_sum",
    "event_active_nearby_flag", "events_nearby_count", "nearest_event_km",
    "event_weighted_intensity", "event_attendance_est_sum_nearby", "event_impact_score",
    "hour_complete_flag", "flow_quality_flag",
    "lat", "lon",
    "hour_end_snapshot_ts_local",
    "num_bikes_available_end_hour",
    "num_docks_available_end_hour",
    "capacity"
}

missing = sorted(list(required_cols - set(df.columns)))
if missing:
    raise Exception(f"{FLOW_TBL} missing required columns: {missing}")

df = df.filter(
    (F.col("lat").between(DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX)) &
    (F.col("lon").between(DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX))
)

df = (
    df
    .withColumn(
        "net_flow",
        (F.col("estimated_arrivals") - F.col("estimated_departures")).cast("double")
    )
    .withColumn("abs_net_flow", F.abs(F.col("net_flow")))
    .withColumn(
        "ts_hour",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.col("date").cast("string"),
                F.format_string("%02d:00:00", F.col("hour"))
            )
        )
    )
)

df_hist = df.filter(
    (F.col("hour_complete_flag") == 1) &
    (F.col("flow_quality_flag") == 1)
)

latest_closed_ts = df_hist.select(F.max("ts_hour").alias("mx")).collect()[0]["mx"]
if latest_closed_ts is None:
    raise Exception("Could not determine latest closed hour.")

print("Latest closed hour:", latest_closed_ts)

start_ts = spark.sql(
    f"SELECT timestamp('{str(latest_closed_ts)}') - INTERVAL {LOOKBACK_HOURS} HOURS AS st"
).collect()[0]["st"]

df_hist = df_hist.filter(
    (F.col("ts_hour") >= F.lit(start_ts)) &
    (F.col("ts_hour") <= F.lit(latest_closed_ts))
)

# ------------------------------------------------------------
# 5) BUILD HISTORICAL FEATURES
# ------------------------------------------------------------
w = Window.partitionBy("station_id").orderBy(F.col("ts_hour"))
roll_w_3h = w.rowsBetween(-3, -1)
roll_w_24h = w.rowsBetween(-24, -1)

df_hist_feat = (
    df_hist
    .withColumn("lag1_net",   F.lag("net_flow", 1).over(w))
    .withColumn("lag2_net",   F.lag("net_flow", 2).over(w))
    .withColumn("lag24_net",  F.lag("net_flow", 24).over(w))
    .withColumn("lag168_net", F.lag("net_flow", 168).over(w))
    .withColumn("roll_mean_3h_net", F.avg("net_flow").over(roll_w_3h))
    .withColumn("roll_std_24h_net", F.stddev("net_flow").over(roll_w_24h))
    .withColumn("station_mean_24h_net", F.avg("net_flow").over(roll_w_24h))
    .withColumn("station_abs_mean_24h_net", F.avg("abs_net_flow").over(roll_w_24h))
    .withColumn("station_std_24h_net", F.stddev("net_flow").over(roll_w_24h))
)

df_last = (
    df_hist_feat
    .filter(F.col("ts_hour") == F.lit(latest_closed_ts))
    .select(
        "station_id", "lat", "lon", "capacity",
        "hour_end_snapshot_ts_local",
        "num_bikes_available_end_hour",
        "num_docks_available_end_hour",
        "net_flow",
        "lag1_net", "lag2_net", "lag24_net", "lag168_net",
        "roll_mean_3h_net", "roll_std_24h_net",
        "station_mean_24h_net", "station_abs_mean_24h_net", "station_std_24h_net"
    )
)

pdf_state = df_last.toPandas()
if pdf_state.empty:
    raise Exception("No station state available for latest closed hour.")

# ------------------------------------------------------------
# 6) READ FUTURE WEATHER
# ------------------------------------------------------------
df_weather_future = spark.table(WEATHER_FUTURE_VIEW).select(
    "year", "month", "day", "hour",
    "weather_ts_local",
    "temperature_2m_c",
    "apparent_temperature_c"
)

pdf_weather_future = df_weather_future.toPandas()

if not pdf_weather_future.empty:
    pdf_weather_future["weather_ts_local"] = pd.to_datetime(pdf_weather_future["weather_ts_local"])
    for c in ["year", "month", "day", "hour"]:
        pdf_weather_future[c] = pd.to_numeric(pdf_weather_future[c], errors="coerce").astype("Int64")
    pdf_weather_future["temperature_2m_c"] = pd.to_numeric(pdf_weather_future["temperature_2m_c"], errors="coerce")
    pdf_weather_future["apparent_temperature_c"] = pd.to_numeric(pdf_weather_future["apparent_temperature_c"], errors="coerce")

print("Future weather rows loaded:", len(pdf_weather_future))

# ------------------------------------------------------------
# 7) PREPARE RECURSIVE STATE
# ------------------------------------------------------------
pdf_state["station_bucket"] = pdf_state["station_id"].map(lambda s: station_to_bucket(s, HASH_BUCKETS))
pdf_state["num_bikes_available_end_hour"] = pd.to_numeric(pdf_state["num_bikes_available_end_hour"], errors="coerce").fillna(0.0)
pdf_state["capacity"] = pd.to_numeric(pdf_state["capacity"], errors="coerce").fillna(0.0)

for c in [
    "lag1_net", "lag2_net", "lag24_net", "lag168_net",
    "roll_mean_3h_net", "roll_std_24h_net",
    "station_mean_24h_net", "station_abs_mean_24h_net", "station_std_24h_net",
    "net_flow", "lat", "lon"
]:
    if c in pdf_state.columns:
        pdf_state[c] = pd.to_numeric(pdf_state[c], errors="coerce").fillna(0.0)

# ------------------------------------------------------------
# 8) OPTIMIZATION: PRECOMPUTE EVENT FEATURES ONCE
# ------------------------------------------------------------
event_features_by_ts = build_precomputed_event_features(
    pdf_state_base=pdf_state,
    pdf_events_detail=pdf_events_detail,
    latest_closed_ts=latest_closed_ts,
    horizon_hours=HORIZON_HOURS
)

print("Precomputed event feature timestamps:", len(event_features_by_ts))

# validation snapshot before forecasting
sample_ts = pd.Timestamp(latest_closed_ts) + pd.Timedelta(hours=1)
if sample_ts in event_features_by_ts:
    sample_ev = event_features_by_ts[sample_ts]
    print("Sample precomputed event columns:")
    print(sample_ev.columns.tolist())

results = []
pdf_state["last_pred_net"] = pdf_state["net_flow"].astype(float)

# ------------------------------------------------------------
# 9) RECURSIVE FORECAST LOOP
# ------------------------------------------------------------
for horizon in range(1, HORIZON_HOURS + 1):
    target_ts = pd.Timestamp(latest_closed_ts) + pd.Timedelta(hours=horizon)

    year = int(target_ts.year)
    month = int(target_ts.month)
    day = int(target_ts.day)
    hour = int(target_ts.hour)
    date = pd.Timestamp(target_ts.date())

    dow_num = int(((target_ts.dayofweek + 1) % 7) + 1)
    is_weekend = 1 if dow_num in [1, 7] else 0

    hour_sin, hour_cos = cyc_hour(hour)
    dow_sin, dow_cos = cyc_dow(dow_num)
    month_sin, month_cos = cyc_month(month)

    step_df = pdf_state.copy()

    # --------------------------------------------------------
    # FUTURE WEATHER
    # --------------------------------------------------------
    wx_step = pdf_weather_future[
        (pdf_weather_future["year"] == year) &
        (pdf_weather_future["month"] == month) &
        (pdf_weather_future["day"] == day) &
        (pdf_weather_future["hour"] == hour)
    ].copy()

    if not wx_step.empty:
        wx_row = wx_step.iloc[0]
        step_df["temperature_2m_celsius"] = float(wx_row["temperature_2m_c"])
        step_df["apparent_temperature_celsius"] = float(wx_row["apparent_temperature_c"])
    else:
        step_df["temperature_2m_celsius"] = 0.0
        step_df["apparent_temperature_celsius"] = 0.0

    # --------------------------------------------------------
    # FUTURE EVENTS FROM PRECOMPUTED DETAIL FEATURES
    # --------------------------------------------------------
    ev_feat = event_features_by_ts.get(target_ts)
    if ev_feat is not None:
        for c in [
            "event_day_flag",
            "events_day_count",
            "event_day_attendance_sum",
            "event_active_nearby_flag",
            "events_nearby_count",
            "nearest_event_km",
            "event_weighted_intensity",
            "event_attendance_est_sum_nearby",
            "event_impact_score",
            "event_spillover_score",
            "event_attendance_sum_03km",
            "event_attendance_sum_06km",
            "event_attendance_sum_15km",
            "event_attendance_sum_30km",
            "primary_event_id",
            "primary_event_name",
            "primary_event_category",
            "primary_event_distance_km",
            "impacting_event_ids",
            "impacting_event_names"
        ]:
            step_df[c] = ev_feat[c].values
    else:
        step_df["event_day_flag"] = 0
        step_df["events_day_count"] = 0
        step_df["event_day_attendance_sum"] = 0.0
        step_df["event_active_nearby_flag"] = 0
        step_df["events_nearby_count"] = 0
        step_df["nearest_event_km"] = 999.0
        step_df["event_weighted_intensity"] = 0.0
        step_df["event_attendance_est_sum_nearby"] = 0.0
        step_df["event_impact_score"] = 0.0
        step_df["event_spillover_score"] = 0.0
        step_df["event_attendance_sum_03km"] = 0.0
        step_df["event_attendance_sum_06km"] = 0.0
        step_df["event_attendance_sum_15km"] = 0.0
        step_df["event_attendance_sum_30km"] = 0.0
        step_df["primary_event_id"] = None
        step_df["primary_event_name"] = None
        step_df["primary_event_category"] = None
        step_df["primary_event_distance_km"] = None
        step_df["impacting_event_ids"] = None
        step_df["impacting_event_names"] = None

    # --------------------------------------------------------
    # CALENDAR / CYCLIC
    # --------------------------------------------------------
    step_df["year"] = year
    step_df["month"] = month
    step_df["day"] = day
    step_df["hour"] = hour
    step_df["date"] = date
    step_df["dow_num"] = dow_num
    step_df["is_weekend"] = is_weekend

    step_df["hour_sin"] = hour_sin
    step_df["hour_cos"] = hour_cos
    step_df["dow_sin"] = dow_sin
    step_df["dow_cos"] = dow_cos
    step_df["month_sin"] = month_sin
    step_df["month_cos"] = month_cos

    # --------------------------------------------------------
    # RECURSIVE FEATURE UPDATE
    # --------------------------------------------------------
    step_df["lag168_net"] = step_df["lag24_net"].fillna(step_df["lag1_net"]).fillna(0.0)
    step_df["lag24_net"] = step_df["lag1_net"].fillna(0.0)
    step_df["lag2_net"] = step_df["lag1_net"].fillna(0.0)
    step_df["lag1_net"] = step_df["last_pred_net"].fillna(0.0)

    step_df["roll_mean_3h_net"] = (
        0.6 * step_df["lag1_net"].astype(float) +
        0.3 * step_df["lag2_net"].astype(float) +
        0.1 * step_df["roll_mean_3h_net"].astype(float)
    )

    step_df["station_mean_24h_net"] = (
        0.9 * step_df["station_mean_24h_net"].astype(float) +
        0.1 * step_df["lag1_net"].astype(float)
    )

    step_df["station_abs_mean_24h_net"] = (
        0.9 * step_df["station_abs_mean_24h_net"].astype(float) +
        0.1 * np.abs(step_df["lag1_net"].astype(float))
    )

    step_df["roll_std_24h_net"] = step_df["roll_std_24h_net"].astype(float).fillna(0.0)
    step_df["station_std_24h_net"] = step_df["station_std_24h_net"].astype(float).fillna(0.0)

    # --------------------------------------------------------
    # ENSURE ALL FEATURES EXIST
    # --------------------------------------------------------
    for c in FEATURES:
        if c not in step_df.columns:
            if c in [
                "station_bucket", "month", "hour", "dow_num", "is_weekend",
                "event_day_flag", "events_day_count", "event_active_nearby_flag", "events_nearby_count"
            ]:
                step_df[c] = 0
            else:
                step_df[c] = 0.0

    # IMPORTANT VALIDATION:
    # The model must still receive exactly the same feature list/order
    X = step_df[FEATURES].astype(np.float32).values
    dmat = xgb.DMatrix(X, feature_names=FEATURES)
    pred = booster.predict(dmat).astype(np.float32)

    step_df["net_pred_full_hour"] = pred
    step_df["net_pred"] = pred

    # --------------------------------------------------------
    # UPDATE PROJECTED BIKES / DOCKS
    # --------------------------------------------------------
    step_df["predicted_bikes_next_hour"] = (
        step_df["num_bikes_available_end_hour"].astype(float) +
        step_df["net_pred"].astype(float)
    )

    step_df["predicted_bikes_next_hour"] = np.clip(
        step_df["predicted_bikes_next_hour"],
        0.0,
        step_df["capacity"].astype(float)
    )

    step_df["predicted_docks_next_hour"] = (
        step_df["capacity"].astype(float) - step_df["predicted_bikes_next_hour"]
    )

    step_df["risk_level"] = [
        risk_label(b, d)
        for b, d in zip(step_df["predicted_bikes_next_hour"], step_df["predicted_docks_next_hour"])
    ]

    step_df["forecast_generated_from_hour"] = pd.Timestamp(latest_closed_ts)
    step_df["target_ts_hour"] = target_ts
    step_df["forecast_horizon_hour"] = horizon
    step_df["model_version"] = "netflow_direct_v3_cyclic_stationtrend"
    step_df["best_model"] = BEST_MODEL

    results.append(step_df[[
        "station_id", "lat", "lon", "capacity",
        "forecast_generated_from_hour", "target_ts_hour", "forecast_horizon_hour",
        "year", "month", "day", "hour", "date",
        "num_bikes_available_end_hour",
        "temperature_2m_celsius", "apparent_temperature_celsius",
        "event_day_flag", "events_day_count", "event_day_attendance_sum",
        "event_active_nearby_flag", "events_nearby_count", "nearest_event_km",
        "event_weighted_intensity", "event_attendance_est_sum_nearby", "event_impact_score",
        "event_spillover_score",
        "event_attendance_sum_03km", "event_attendance_sum_06km",
        "event_attendance_sum_15km", "event_attendance_sum_30km",
        "primary_event_id", "primary_event_name", "primary_event_category",
        "primary_event_distance_km",
        "impacting_event_ids", "impacting_event_names",
        "net_pred_full_hour", "net_pred",
        "predicted_bikes_next_hour", "predicted_docks_next_hour",
        "risk_level",
        "model_version", "best_model"
    ]].copy())

    pdf_state["last_pred_net"] = step_df["net_pred"].values
    pdf_state["num_bikes_available_end_hour"] = step_df["predicted_bikes_next_hour"].values
    pdf_state["num_docks_available_end_hour"] = step_df["predicted_docks_next_hour"].values

    pdf_state["lag168_net"] = step_df["lag168_net"].values
    pdf_state["lag24_net"] = step_df["lag24_net"].values
    pdf_state["lag2_net"] = step_df["lag2_net"].values
    pdf_state["lag1_net"] = step_df["lag1_net"].values
    pdf_state["roll_mean_3h_net"] = step_df["roll_mean_3h_net"].values
    pdf_state["roll_std_24h_net"] = step_df["roll_std_24h_net"].values
    pdf_state["station_mean_24h_net"] = step_df["station_mean_24h_net"].values
    pdf_state["station_abs_mean_24h_net"] = step_df["station_abs_mean_24h_net"].values
    pdf_state["station_std_24h_net"] = step_df["station_std_24h_net"].values

# ------------------------------------------------------------
# 10) WRITE OUTPUT
# ------------------------------------------------------------
forecast_pd = pd.concat(results, ignore_index=True)

for c in ["forecast_generated_from_hour", "target_ts_hour", "date"]:
    if c in forecast_pd.columns:
        forecast_pd[c] = pd.to_datetime(forecast_pd[c], errors="coerce")

forecast_pd["forecast_generated_from_hour"] = forecast_pd["forecast_generated_from_hour"].dt.strftime("%Y-%m-%d %H:%M:%S")
forecast_pd["target_ts_hour"] = forecast_pd["target_ts_hour"].dt.strftime("%Y-%m-%d %H:%M:%S")
forecast_pd["date"] = forecast_pd["date"].dt.strftime("%Y-%m-%d")

forecast_sdf = spark.createDataFrame(forecast_pd.to_dict("records"))

forecast_sdf = (
    forecast_sdf
    .drop("forecast_generated_from_hour")
    .withColumn("forecast_generated_from_hour", F.lit(latest_closed_ts).cast("timestamp"))
    .withColumn("target_ts_hour", F.to_timestamp("target_ts_hour"))
    .withColumn("date", F.to_date("date"))
    .withColumn("scored_at_utc", F.current_timestamp())
)

print("Final schema:")
forecast_sdf.printSchema()

(
    forecast_sdf
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MULTI_FORECAST_OUT_TBL)
)

print("✅ Multi-hour forecast table created:", MULTI_FORECAST_OUT_TBL)
print("Rows:", forecast_sdf.count())

display(
    forecast_sdf.orderBy("station_id", "forecast_horizon_hour").limit(50)
)

## Outputs

This notebook produces the following table:

- `workspace.default.current_station_predictions_multihour_netflow_direct_v3_cyclic_stationtrend`

### Output Characteristics
The table contains one row per:

- station
- future forecast hour

across a **168-hour horizon**.

### Main Output Fields
The forecast output includes:

- station metadata
- forecast horizon hour
- forecast target timestamp
- projected weather conditions
- projected event-related influence
- predicted full-hour net flow
- predicted bikes available
- predicted docks available
- station-level risk label

Examples of key columns include:
- `forecast_generated_from_hour`
- `target_ts_hour`
- `forecast_horizon_hour`
- `net_pred_full_hour`
- `net_pred`
- `predicted_bikes_next_hour`
- `predicted_docks_next_hour`
- `risk_level`

This dataset is designed for multi-hour operational planning and station risk monitoring.

## Key Insights and Summary

### 1. This notebook extends forecasting from short-term to planning horizon
Unlike the 1-hour prediction engine, this notebook projects demand dynamics across an entire week, making it suitable for medium-term operational planning.

---

### 2. The forecast is recursive by design
Each predicted hour becomes part of the input state for the next forecast step. This allows the model to simulate how station conditions evolve over time rather than scoring each hour independently.

This makes the output more operationally realistic.

---

### 3. Future weather and future events are incorporated explicitly
The engine uses:
- future hourly weather conditions
- future event timing and attendance
- event spillover influence by station

This enables the forecast to reflect known future external drivers rather than relying only on historical momentum.

---

### 4. Precomputed event features improve efficiency and consistency
Because event influence is calculated once for all forecast hours before the recursive loop, the notebook improves performance while maintaining consistent station-event logic throughout the horizon.

---

### 5. Predictions are operationally constrained
Projected bike counts are clipped by station capacity, which ensures:
- no negative bike counts
- no over-capacity forecasts
- realistic projected docks availability

This keeps the recursive simulation physically consistent.

---

### 6. Risk labels improve multi-hour interpretability
By translating predicted station states into operational categories such as:
- `NORMAL`
- `LOW_BIKES`
- `LOW_DOCKS`
- `CRITICAL_EMPTY`
- `CRITICAL_FULL`

the notebook makes long-horizon forecasts easier to interpret and act upon.

---

### 7. Business relevance
This notebook supports:
- proactive bike rebalancing
- medium-term station risk monitoring
- operational planning for high-demand periods
- anticipation of event-driven demand shocks

In practical terms, this notebook is the **1-week forecasting engine** that transforms the trained model into a forward-looking operational planning tool.